In [ ]:
# collect_handeye_points.py
import cv2
import numpy as np
import os
import json

H_FILE = "calibration/output/homography_pixel_to_shelf.npy"
OUT = "calibration/output/handeye_points.json"
cap = cv2.VideoCapture(0)

if not os.path.exists(H_FILE):
    print("Homography missing. Run compute_homography.py first.")
    exit(1)
H = np.load(H_FILE)

handeye_pairs = []  # list of dicts: {"cam": [x,y], "robot":[x,y]}

def pixel_to_shelf(u, v, H):
    p = np.array([u, v, 1.0])
    mapped = H.dot(p)
    mapped /= mapped[2]
    return float(mapped[0]), float(mapped[1])

clicked = None
def mouse_cb(event, x, y, flags, param):
    global clicked
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked = (x, y)
        print("Clicked pixel", clicked)

cv2.namedWindow("Collect hand-eye")
cv2.setMouseCallback("Collect hand-eye", mouse_cb)

print("Click points in the video where you want correspondences.")
print("For each click: the script will show shelf coords. Move the robot end-effector to that location and then press 'r' to record robot coords input manually.")
print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    display = frame.copy()
    if clicked is not None:
        u, v = clicked
        cv2.circle(display, (u, v), 6, (0,255,0), -1)
        sx, sy = pixel_to_shelf(u, v, H)
        cv2.putText(display, f"shelf X={sx:.3f} Y={sy:.3f}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,200,200), 2)
    cv2.imshow("Collect hand-eye", display)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    if key == ord('r') and clicked is not None:
        sx, sy = pixel_to_shelf(clicked[0], clicked[1], H)
        print(f"Please move robot to shelf coords X={sx:.3f}, Y={sy:.3f} and then enter robot coords as 'x y' (meters) now.")
        inp = input("robot_x robot_y: ")
        try:
            rx, ry = map(float, inp.strip().split())
            handeye_pairs.append({"cam": [sx, sy], "robot":[rx, ry]})
            print("Recorded pair:", handeye_pairs[-1])
            # save incrementally
            with open(OUT, "w") as f:
                json.dump(handeye_pairs, f, indent=2)
            print("Saved to", OUT)
        except Exception as e:
            print("Invalid input, skipped.", e)

cap.release()
cv2.destroyAllWindows()
